In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import json

gap=pd.read_csv("../GAP_data/gap_scores.csv")
bounds=gpd.read_file("../boundaryfile.geojson")
merged=bounds.merge(gap,left_on="LAD24NM",right_on="borough",how="left")
print(merged.shape)
print(merged[["LAD24NM","class"]].head())

In [ ]:
merged_json=json.loads(merged.to_json())

In [ ]:
#chloropleth
fig=px.choropleth_mapbox(merged,geojson=merged_json,locations=merged.index,
    color="class",
    hover_name="LAD24NM",
    hover_data={"demand_score": ":.1f","inactive": ":.1f","readiness_opportunity_pct": ":.1f","sessions": True,"venues": True,"note": True,"class": False,},
    mapbox_style="carto-positron",center={"lat": 51.5074,"lon": -0.1278},
    zoom=8.5,opacity=0.7,title="London physical activity engagement gap map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [ ]:
# equity
pop=pd.read_csv("../ActiveLives_Data/poplprofile_v2_clean.csv")
gap=pd.read_csv("../GAP_data/gap_scores.csv")

imd=pop[pop["demographic_group"] == "IMD10"].copy()
deprived=imd[imd["category"].isin(["Decile 1 (most deprived)","Decile 2","Decile 3"])]
deprived_summary=deprived.groupby("borough").agg(deprived_inactive=("pct_inactive","mean"),deprived_sample=("respondents","sum")).reset_index()
gap_equity=gap.merge(deprived_summary,on="borough",how="left")
print(gap_equity[["borough","class","demand_score","inactive","deprived_inactive"]].sort_values("deprived_inactive",ascending=False))

gap_equity.to_csv("../GAP_data/gap_scores_equity.csv",index=False)
print("saved gap_scores_equity.csv")

In [ ]:
# equity gap 
gap_equity=pd.read_csv("../GAP_data/gap_scores_equity.csv")
gap_equity["equity_gap"]=gap_equity["deprived_inactive"] - gap_equity["inactive"]
print(gap_equity[["borough","class","demand_score","inactive","deprived_inactive","equity_gap"]].sort_values("equity_gap",ascending=False))

In [ ]:
gap_equity["equity_flag"]=pd.qcut(gap_equity["equity_gap"],3,labels=["low gap","moderate gap","high gap"])
print(gap_equity[["borough","class","equity_gap","equity_flag"]].sort_values("equity_gap",ascending=False))
gap_equity.to_csv("../GAP_data/gap_scores_equity.csv",index=False)

In [ ]:
london_boroughs=gap["borough"].unique().tolist()
merged=merged[merged["LAD24NM"].isin(london_boroughs) | (merged["LAD24NM"] == "City of London")]
print(merged.shape)  

In [ ]:
# classification and equity
gap=pd.read_csv("../GAP_data/gap_scores_equity.csv")
bounds=gpd.read_file("../boundaryfile.geojson")
london_boroughs=gap["borough"].unique().tolist()
bounds=bounds[bounds["LAD24NM"].isin(london_boroughs) | (bounds["LAD24NM"] == "City of London")]
merged=bounds.merge(gap,left_on="LAD24NM",right_on="borough",how="left")
merged_json=json.loads(merged.to_json())

fig1=px.choropleth_mapbox(merged,geojson=merged_json,locations=merged.index,color="class",hover_name="LAD24NM",hover_data={"demand_score": ":.1f","inactive": ":.1f","sessions": True,"class": False},mapbox_style="carto-positron",center={"lat": 51.5074,"lon": -0.1278},zoom=8.5,opacity=0.75,title="Supply vs demand classification")
fig1.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig1.write_html("../GAP_data/gap_map_classification.html")


fig2=px.choropleth_mapbox(merged,geojson=merged_json,locations=merged.index,color="equity_flag",hover_name="LAD24NM",
    hover_data={"inactive": ":.1f","deprived_inactive": ":.1f","equity_gap": ":.1f","equity_flag": False},mapbox_style="carto-positron",center={"lat": 51.5074,"lon": -0.1278},zoom=8.5,
    opacity=0.75,color_discrete_map={"high gap":"#B22222","moderate gap":"#DAA520","low gap":"#90EE90"},title="Equity gap: deprived vs overall inactivity")
fig2.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig2.write_html("../GAP_data/gap_map_equity.html")


In [ ]:
print(merged[["LAD24NM","class","equity_flag"]].dropna().shape)
print(merged[["LAD24NM","class","equity_flag"]].head(10))

(32, 3)
                LAD24NM                        class equity_flag
0        City of London                          NaN         NaN
1  Barking and Dagenham  well-served (moderate need)     low gap
2                Barnet                      average    high gap
3                Bexley            adequately served    high gap
4                 Brent              under-monitored     low gap
5               Bromley               genuine desert    high gap
6                Camden         low need, low supply     low gap
7               Croydon              blind spot risk     low gap
8                Ealing                  well-served     low gap
9               Enfield                   blind spot    high gap


In [ ]:
df=pd.read_csv("../OPENACTIVE_MERGED.csv")
gap=pd.read_csv("../GAP_data/gap_scores_equity.csv")

# % of each borough's sessions by activity type
mix=df.groupby(["borough","activity_type"])["session_count"].sum().reset_index()
mix_pivot=mix.pivot(index="borough",columns="activity_type",values="session_count").fillna(0)
mix_pct=mix_pivot.div(mix_pivot.sum(axis=1),axis=0) * 100

# sessions from the single but huge venue
venue_totals=df.groupby(["borough","location_name"])["session_count"].sum().reset_index()
top_venue=venue_totals.sort_values("session_count",ascending=False).drop_duplicates("borough")
borough_totals=df.groupby("borough")["session_count"].sum().reset_index(name="total")
top_venue=top_venue.merge(borough_totals,on="borough")
top_venue["top_venue_share"]=top_venue["session_count"] / top_venue["total"] * 100

# using median (tried mean but comes skewed)
avg_price=df[df["price_status"] == "known"].groupby("borough")["price_gbp"].median().reset_index(name="avg_price")
price_sample_size=df[df["price_status"] == "known"].groupby("borough").size().reset_index(name="price_sample_size")
avg_price=avg_price.merge(price_sample_size,on="borough")
avg_price["price_reliable"]=avg_price["price_sample_size"] >= 20
better_share=df.groupby("borough")["provider_group"].apply(lambda x: (x == "Better").mean() * 100).reset_index(name="pct_better")
profile=gap[["borough","class","demand_score","inactive","sessions","equity_gap"]].copy()
profile=profile.merge(top_venue[["borough","top_venue_share"]],on="borough",how="left")
profile=profile.merge(avg_price,on="borough",how="left")
profile=profile.merge(better_share,on="borough",how="left")

print(profile.sort_values("demand_score",ascending=False))
profile.to_csv("../GAP_data/borough_profile.csv",index=False)

In [ ]:
richmond_check=df[(df["borough"]=="Richmond upon Thames") & (df["price_status"]=="known")]
print(len(richmond_check))
print(df[df["borough"]=="Richmond upon Thames"]["price_status"].value_counts())

In [ ]:

# borough against average
wellserved=profile[profile["class"].isin(["well-served","well-served (moderate need)"])]
baseline=wellserved[["top_venue_share","avg_price","pct_better"]].mean()

print("well-served baseline:")
print(baseline)
flagged=profile[profile["class"].isin(["genuine desert","blind spot","under-monitored","emerging desert"])].copy()
flagged["venue_diff"]=flagged["top_venue_share"] - baseline["top_venue_share"]
flagged["price_diff"]=flagged["avg_price"] - baseline["avg_price"]
flagged["better_diff"]=flagged["pct_better"] - baseline["pct_better"]
print(flagged[["borough","class","venue_diff","price_diff","better_diff"]].sort_values("venue_diff",ascending=False))

In [ ]:
richmond_check=df[(df["borough"]=="Richmond upon Thames") & (df["price_status"]=="known")]
print("richmond known-price rows:",len(richmond_check))
print(df[df["borough"]=="Richmond upon Thames"]["price_status"].value_counts())

In [ ]:
profile=pd.read_csv("../GAP_data/borough_profile.csv")
wellserved=profile[profile["class"].isin(["well-served","well-served (moderate need)"])]
baseline=wellserved[["top_venue_share","avg_price","pct_better"]].mean()
print("well-served baseline:")
print(baseline)
flagged=profile[profile["class"].isin(["genuine desert","blind spot","under-monitored","emerging desert"])].copy()
flagged["venue_diff"]=flagged["top_venue_share"] - baseline["top_venue_share"]
flagged["price_diff"]=flagged["avg_price"] - baseline["avg_price"]
flagged["better_diff"]=flagged["pct_better"] - baseline["pct_better"]
print(flagged[["borough","class","venue_diff","price_diff","better_diff"]].sort_values("venue_diff",ascending=False))

flagged.to_csv("../GAP_data/borough_profile_diagnostics.csv",index=False)
print("saved borough_profile_diagnostics.csv")

In [ ]:
# free abd paid 
free_share=df.groupby("borough")["price_status"].apply(lambda x: (x == "free").mean() * 100).reset_index(name="pct_free")

profile=pd.read_csv("../GAP_data/borough_profile.csv")
profile=profile.merge(free_share,on="borough",how="left")
print(profile[["borough","class","avg_price","pct_free","equity_gap"]].sort_values("pct_free",ascending=False))
profile.to_csv("../GAP_data/borough_profile.csv",index=False)

In [ ]:
df["price_for_avg"]=df["price_gbp"].where(df["price_status"] != "free",0)
blended_price=df[df["price_status"].isin(["known","free"])].groupby("borough")["price_for_avg"].mean().reset_index(name="blended_avg_price")
profile=profile.merge(blended_price,on="borough",how="left")
print(profile[["borough","avg_price","blended_avg_price","pct_free"]].sort_values("blended_avg_price",ascending=False))

In [ ]:
profile["price_note"]=""
profile.loc[profile["borough"]=="Richmond upon Thames","price_note"]="100% free supply — no paid sessions to average"
profile.loc[profile["borough"]=="Redbridge","price_note"]="zero OpenActive sessions — no price data exists"

profile.to_csv("../GAP_data/borough_profile.csv",index=False)
print(profile[["borough","avg_price","blended_avg_price","price_note"]].tail(10))